# PPDBTAP Service Test Notebook

Automated tests for the ppdb TAP service.

## Imports

In [ ]:
from pyvo.dal.tap import TAPService
from astropy.table import Table
from lsst.rsp.utils import get_access_token
from lsst.rsp import RSPClient, get_service_url
import requests
from lsst.rsp import RSPDiscovery

In [ ]:
discovery = RSPDiscovery("prompt")
service = discovery.get_tap_client()
assert service is not None
print(f"TAP service URL: {service.baseurl}")

## Check VOSI Tables

In [ ]:
tables = service.tables
print(f"Found {len(tables)} tables")

## Schema Discovery - List Schemas

In [ ]:
results = service.search("SELECT schema_name, description FROM TAP_SCHEMA.schemas")
assert len(results) > 0
print(f"Found {len(results)} schemas")
results.to_table()

## Schema Discovery - List Tables

In [ ]:
results = service.search("SELECT table_name, description FROM TAP_SCHEMA.tables WHERE schema_name = 'ppdb'")
assert len(results) > 0
print(f"Found {len(results)} tables in ppdb")
results.to_table()

## Schema Discovery - List Columns

In [ ]:
results = service.search(
    "SELECT column_name, datatype, unit FROM TAP_SCHEMA.columns "
    "WHERE table_name = 'ppdb.DiaObject' "
    "ORDER BY column_name"
)
assert len(results) > 0
print(f"Found {len(results)} columns in Object table")

## Synchronous Query - Small Result Set (TOP 10)

In [ ]:
query = """
SELECT TOP 1 *
FROM ppdb.DiaObject
WHERE diaObjectId=26
"""
results = service.search(query)
assert len(results) <= 10
print(f"Retrieved {len(results)} objects")
results.to_table()

## Asynchronous Query - Small Result Set (TOP 10 with 30 second timeout)

In [ ]:
query = """
SELECT TOP 1 *
FROM ppdb.DiaObject
WHERE diaObjectId=26
"""
job = service.submit_job(query)
job.run()
job.wait(phases=['COMPLETED', 'ERROR'], timeout=30)
if job.phase not in ('COMPLETED', 'ERROR'):
    raise TimeoutError(f"Job timed out after 2 minutes. Current phase: {job.phase}")
if job.phase == 'ERROR':
    job.raise_if_error()
results = job.fetch_result()
results.to_table()